# Optic-X: Handwritten Character Recognition from Scratch
### Multiclass Neural Network (35 Classes: Uppercase A-Z, Digits 1-9) in Pure NumPy

**Author:** Shivanesh V  
**Assignment:** ZenteiQ AI Hub - AI/ML Engineer Internship Assessment  

---
## 1. Problem Statement & Scope
The objective is to implement a Deep Neural Network strictly from scratch using **only NumPy** (without TensorFlow, PyTorch, Keras, or autograd tools) to classify handwritten characters into **35 target classes**:
- **Digits 1–9** (9 classes, excluding 0)
- **Uppercase letters A–Z** (26 classes)

The assignment mandates a strict sequential execution order for the training loop:
1. Forward propagation
2. Backpropagation
3. ReLU/Tanh activation
4. Softmax output layer
5. Cross-entropy loss
6. Gradient descent update
Followed by a validation loop, confusion matrix generation, and deep-dive error analysis.

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
sys.path.append('..')

from src.dataset import (
    CLASS_CHARS, CHAR_TO_IDX, IDX_TO_CHAR,
    load_and_preprocess_emnist, train_val_test_split, one_hot_encode
)
from src.model import NeuralNetwork
from src.optimizer import SGDOptimizer
from src.loss import CategoricalCrossEntropy
from src.metrics import (
    compute_accuracy, compute_confusion_matrix,
    compute_classification_metrics, find_misclassified_samples
)

print(f"Target Classes ({len(CLASS_CHARS)}): {CLASS_CHARS}")

## 2. Dataset Loading & Preprocessing
### The EMNIST Transposition Quirk
Raw EMNIST images are stored transposed (Fortran/column-major format). Without reshaping to `(28, 28).T`, the characters appear sideways and flipped. Our custom pipeline restores upright orientation and scales pixel values to `[0.0, 1.0]`.

In [2]:
# Load cached test partition
data = np.load('../artifacts/test_data.npz')
X_test, y_test = data['X_test'], data['y_test']
print(f"Loaded test samples: {X_test.shape[0]}, Features: {X_test.shape[1]}")

# Visualize sample characters to confirm correct upright orientation
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    idx = i * 20
    img = X_test[idx].reshape(28, 28)
    char = IDX_TO_CHAR[int(y_test[idx])]
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Label: '{char}'", fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Network Architecture & Mathematical Design
The network uses a hierarchical feed-forward topology: `784 -> 256 -> 128 -> 35`.
- **Layer 1 (784 -> 256)**: Captures primitive stroke edges, loops, and diagonal angles.
- **Layer 2 (256 -> 128)**: Combines strokes into character sub-components.
- **Layer 3 (128 -> 35)**: Class-specific logits normalized with numerically stable Softmax.

**Weight Initialization:** He (Kaiming) normal initialization ($W \sim \mathcal{N}(0, \sqrt{2/n_{in}})$) is used for ReLU hidden layers to preserve activation variance across layers.

In [3]:
# Restore trained model weights
model = NeuralNetwork(layer_dims=[784, 256, 128, 35], activation='relu')
model.load_weights('../artifacts/model_weights.npz')
print("Weights loaded successfully.")

## 4. Test Set Evaluation & Metrics
We evaluate the model on the unseen test partition of 5,250 samples.

In [4]:
y_pred_probs = model.forward(X_test)
y_pred = np.argmax(y_pred_probs, axis=-1)

test_acc = compute_accuracy(y_pred, y_test)
cm = compute_confusion_matrix(y_pred, y_test, num_classes=len(CLASS_CHARS))
metrics = compute_classification_metrics(cm)

print("=" * 45)
print(f"Test Accuracy:    {test_acc * 100:.2f}%")
print(f"Macro Precision:  {metrics['macro_precision'] * 100:.2f}%")
print(f"Macro Recall:     {metrics['macro_recall'] * 100:.2f}%")
print(f"Macro F1-Score:   {metrics['macro_f1'] * 100:.2f}%")
print("=" * 45)

## 5. Confusion Matrix Analysis (35x35)
The confusion matrix displays the distribution of predictions across all 35 classes (digits 1–9 and letters A–Z).

In [5]:
fig, ax = plt.subplots(figsize=(12, 10), dpi=150)
cax = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("35-Class Confusion Matrix", fontsize=15, pad=12)
plt.colorbar(cax, fraction=0.046, pad=0.04)

ticks = np.arange(len(CLASS_CHARS))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(CLASS_CHARS, fontsize=7)
ax.set_yticklabels(CLASS_CHARS, fontsize=7)
ax.set_ylabel("True Class", fontsize=11, fontweight='bold')
ax.set_xlabel("Predicted Class", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Deep-Dive Error Analysis (Top Misclassified Cases)
Examining misclassified characters reveals genuine typographical ambiguities and natural stroke variations in human handwriting.

In [6]:
errors = find_misclassified_samples(X_test, y_test, y_pred_probs, top_k=5)

fig, axes = plt.subplots(1, len(errors), figsize=(15, 3.5))
for i, err in enumerate(errors):
    ax = axes[i]
    img = err['image'].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(
        f"True: '{err['true_char']}'\nPred: '{err['pred_char']}' ({err['pred_confidence'] * 100:.1f}%)",
        fontsize=11, color='red', fontweight='bold'
    )
plt.tight_layout()
plt.show()

for i, err in enumerate(errors, 1):
    print(f"Case {i}: True '{err['true_char']}' predicted as '{err['pred_char']}' with {err['pred_confidence']*100:.1f}% confidence.")